In [ ]:
import subprocess, sys
!pip install facenet-pytorch --no-deps

In [ ]:
import os, random, warnings, time
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import torchvision.models as models
from facenet_pytorch import MTCNN
from sklearn.metrics import (classification_report, confusion_matrix,f1_score, roc_auc_score, accuracy_score)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'AMP    : {USE_AMP}')


LABEL_BASE = Path('/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/Labels')
VIDEO_BASE = Path('/kaggle/input/datasets/olgaparfenova/daisee/DAiSEE/DataSet')
TRAIN_CSV = LABEL_BASE / 'TrainLabels.csv'
VAL_CSV = LABEL_BASE / 'ValidationLabels.csv'
TEST_CSV = LABEL_BASE / 'TestLabels.csv'

N_FRAMES = 16       
IMG_SIZE = 224      
BATCH_SIZE = 8         
ACCUM_STEPS = 4        
LR_HEAD = 1e-4      
LR_FULL = 1e-5      
PHASE1_EPOCHS = 5         
PHASE2_EPOCHS = 30        
PATIENCE = 8         
DROPOUT = 0.5
LSTM_HIDDEN = 512
LSTM_LAYERS = 2
FL_ALPHA = 0.25
FL_GAMMA = 2.0
CLASSES = ['Engaged', 'Distracted']

print('\nConfig ready.')
print(f'  N_FRAMES={N_FRAMES}  IMG_SIZE={IMG_SIZE}')
print(f'  Effective batch size = {BATCH_SIZE} × {ACCUM_STEPS} = {BATCH_SIZE*ACCUM_STEPS}')
print(f'  Phase-1: {PHASE1_EPOCHS} epochs (frozen backbone, lr={LR_HEAD})')
print(f'  Phase-2: up to {PHASE2_EPOCHS} epochs (full network, lr={LR_FULL})')

In [ ]:
def relabel_daisee(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    out = df[['ClipID']].copy()
    out['label'] = -1

    m1 = df['Engagement'] > 2
    m2 = df['Engagement'] == 2
    m2a = m2 & (df['Confusion'] == 0) & ((df['Frustration'] != 0) | (df['Boredom'] != 0))
    m2b = m2 & ~m2a
    m3 = ~(m1 | m2)

    out.loc[m1,  'label'] = 0   # Engaged
    out.loc[m2b, 'label'] = 0
    out.loc[m2a, 'label'] = 1   # Distracted
    out.loc[m3,  'label'] = 1

    total = len(out)
    vc    = out['label'].value_counts().sort_index()
    print(f'\n{Path(csv_path).name}  (n={total:,})')
    for lbl, cls in enumerate(CLASSES):
        n = vc.get(lbl, 0)
        print(f'  {cls:12s}: {n:>5,}  ({100*n/total:.1f}%)')

    assert (out['label'] == -1).sum() == 0, 'Unresolved labels found!'
    return out

train_df = relabel_daisee(TRAIN_CSV)
val_df   = relabel_daisee(VAL_CSV)
test_df  = relabel_daisee(TEST_CSV)

In [ ]:
print('Scanning video directory (rglob)…', end=' ', flush=True)
t0 = time.time()
VIDEO_LOOKUP: dict[str, Path] = {}
for ext in ('*.avi', '*.mp4'):
    VIDEO_LOOKUP.update({p.name: p for p in VIDEO_BASE.rglob(ext)})
print(f'found {len(VIDEO_LOOKUP):,} files in {time.time()-t0:.1f}s')

def resolve_path(clip_id: str) -> Path | None:
    fname = Path(clip_id).name
    if fname in VIDEO_LOOKUP:
        return VIDEO_LOOKUP[fname]
    full = VIDEO_BASE / clip_id
    return full if full.exists() else None

for split_name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    found = sum(resolve_path(c) is not None for c in df['ClipID'])
    pct   = 100 * found / len(df)
    status = 'Ok' if pct > 95 else 'Disclaimer'
    print(f'{status} {split_name}: {found}/{len(df)} clips resolved ({pct:.1f}%)')

In [ ]:
_MTCNN = MTCNN(
    image_size=IMG_SIZE,
    margin=20,           
    min_face_size=40,    
    thresholds=[0.6, 0.7, 0.7],
    keep_all=False,      
    device='cpu',        
    post_process=False,  
)


def crop_face(frame_rgb: np.ndarray) -> np.ndarray:
    from PIL import Image
    pil_img = Image.fromarray(frame_rgb)
    try:
        box, _ = _MTCNN.detect(pil_img)
        if box is not None and len(box) > 0:
            x1, y1, x2, y2 = [int(v) for v in box[0]]
            h, w = frame_rgb.shape[:2]
            margin = 20
            x1 = max(0, x1 - margin)
            y1 = max(0, y1 - margin)
            x2 = min(w, x2 + margin)
            y2 = min(h, y2 + margin)
            if x2 > x1 and y2 > y1:
                crop = frame_rgb[y1:y2, x1:x2]
                return cv2.resize(crop, (IMG_SIZE, IMG_SIZE),
                                  interpolation=cv2.INTER_LINEAR)
    except Exception:
        pass
  
    return cv2.resize(frame_rgb, (IMG_SIZE, IMG_SIZE),
                      interpolation=cv2.INTER_LINEAR)

print('MTCNN face detector initialised (CPU, fallback=full-frame).')

_blank = np.zeros((480, 640, 3), dtype=np.uint8)
_out = crop_face(_blank)
assert _out.shape == (IMG_SIZE, IMG_SIZE, 3), f'Unexpected shape: {_out.shape}'
print(f'Smoke test passed → output shape {_out.shape}')

In [ ]:
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

TRAIN_TF = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.03),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    T.ToTensor(),
    T.Normalize(_MEAN, _STD),
])

EVAL_TF = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize(_MEAN, _STD),
])

In [ ]:
def load_clip_frames(
    path: Path,
    n_frames: int = N_FRAMES,
    augment: bool = False,
    use_face: bool = True,
) -> torch.Tensor | None:
    cap   = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total < 1:
        cap.release()
        return None
    
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    if augment and total > n_frames:
        jitter  = max(1, int(0.08 * total))
        offsets = np.random.randint(-jitter, jitter + 1, size=n_frames)
        indices = np.clip(indices + offsets, 0, total - 1)

    tf = TRAIN_TF if augment else EVAL_TF
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            cap.release()
            return None  

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        if use_face:
            frame_rgb = crop_face(frame_rgb)   
        else:
            frame_rgb = cv2.resize(
                frame_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR
            )

        frames.append(tf(frame_rgb))

    cap.release()
    return torch.stack(frames)  

In [ ]:
class DAiSEEDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        n_frames: int = N_FRAMES,
        augment: bool = False,
        use_face: bool = True,
    ):
        self.n_frames = n_frames
        self.augment = augment
        self.use_face = use_face
        self._zero = torch.zeros(n_frames, 3, IMG_SIZE, IMG_SIZE)

        records, missing = [], 0
        for _, row in df.iterrows():
            p = resolve_path(str(row['ClipID']))
            if p is None:
                missing += 1
            else:
                records.append({'path': p, 'label': int(row['label'])})

        self.records = records
        print(f'  [{"train" if augment else "eval "}]  '
              f'{len(records):,} clips  |  '
              f'{missing} missing  |  '
              f'face={use_face}  aug={augment}')

    
    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int):
        rec    = self.records[idx]
        frames = load_clip_frames(
            rec['path'], self.n_frames,
            augment=self.augment, use_face=self.use_face
        )
        if frames is None:
            frames = self._zero
        return frames, rec['label']

    def get_labels(self) -> list[int]:
        return [r['label'] for r in self.records]

print('Building datasets (face detection enabled)…')
train_ds = DAiSEEDataset(train_df,augment=True,use_face=True)
val_ds = DAiSEEDataset(val_df,augment=False,use_face=True)
test_ds = DAiSEEDataset(test_df,augment=False,use_face=True)

In [ ]:
def make_balanced_sampler(dataset: DAiSEEDataset) -> WeightedRandomSampler:
    """Each batch receives ~50% Engaged / 50% Distracted samples."""
    labels = dataset.get_labels()
    counts = Counter(labels)
    weights = {c: 1.0 / counts[c] for c in counts}
    sample_w = torch.tensor([weights[l] for l in labels], dtype=torch.double)
    return WeightedRandomSampler(sample_w, num_samples=len(labels), replacement=True)


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=make_balanced_sampler(train_ds),
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True,
)

print(f'Loaders ready.  train={len(train_loader)} batches  '
      f'val={len(val_loader)}  test={len(test_loader)}')


_lbl_sample = []
for _, lb in train_loader:
    _lbl_sample.extend(lb.tolist())
    if len(_lbl_sample) >= 200: break
_vc = Counter(_lbl_sample)
print(f'Batch balance check (~200 samples): '
      f'Engaged={_vc[0]} ({100*_vc[0]/sum(_vc.values()):.0f}%)  '
      f'Distracted={_vc[1]} ({100*_vc[1]/sum(_vc.values()):.0f}%)')

In [ ]:
class TemporalAttention(nn.Module):
    
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        scores = self.attn(x)          # (B, T, 1)
        weights = torch.softmax(scores, dim=1)  # (B, T, 1)
        context = (weights * x).sum(dim=1)      # (B, D)
        return context

In [ ]:
class EfficientNetBiLSTM(nn.Module):

    FEAT_DIM = 1536          
    LSTM_OUT  = LSTM_HIDDEN * 2  

    def __init__(
        self,
        n_classes: int = 2,
        lstm_hidden: int = LSTM_HIDDEN,
        lstm_layers: int = LSTM_LAYERS,
        dropout: float = DROPOUT,
    ):
        super().__init__()

        
        effnet = models.efficientnet_b3(weights='DEFAULT')
        
        self.backbone = effnet.features   # output: (B, 1536, H', W')
        self.pool = nn.AdaptiveAvgPool2d(1)   # → (B, 1536, 1, 1)

        
        self.lstm = nn.LSTM(
            input_size=self.FEAT_DIM,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.attention = TemporalAttention(self.LSTM_OUT)

        
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.LSTM_OUT),
            nn.Dropout(dropout),
            nn.Linear(self.LSTM_OUT, 512),
            nn.GELU(),
            nn.Dropout(dropout * 0.6),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.4),
            nn.Linear(256, n_classes),
        )

        self._init_weights()

    def _init_weights(self):
        """Xavier-init LSTM and linear layers."""
        for name, p in self.lstm.named_parameters():
            if 'weight_ih' in name: nn.init.xavier_uniform_(p)
            elif 'weight_hh' in name: nn.init.orthogonal_(p)
            elif 'bias' in name: nn.init.zeros_(p)
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    
    def freeze_backbone(self):
        """Phase 1: freeze EfficientNet, only train LSTM + head."""
        for p in self.backbone.parameters():
            p.requires_grad = False
        print('Backbone FROZEN. Training LSTM + attention + head only.')

    def unfreeze_backbone(self):
        """Phase 2: unfreeze entire network for end-to-end fine-tuning."""
        for p in self.backbone.parameters():
            p.requires_grad = True
        print('Backbone UNFROZEN. Full network fine-tuning.')

    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)       # (B*T, C, H, W)
        x = self.backbone(x)              # (B*T, 1536, H', W')
        x = self.pool(x).flatten(1)       # (B*T, 1536)
        x = x.view(B, T, -1)             # (B, T, 1536)        
        x, _ = self.lstm(x)              # (B, T, 1024)
        x    = self.attention(x)          # (B, 1024)

        return self.classifier(x)         # (B, 2)


def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, train

model = EfficientNetBiLSTM().to(DEVICE)
total, train = count_params(model)
print(f'\nTotal params     : {total/1e6:.2f}M')
print(f'Trainable params : {train/1e6:.2f}M')

with torch.no_grad():
    _dummy = torch.zeros(2, N_FRAMES, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    _out   = model(_dummy)
    assert _out.shape == (2, 2), f'Unexpected output shape: {_out.shape}'
    print(f'Forward pass OK  : output shape {_out.shape}')
del _dummy, _out

In [ ]:
class FocalLoss(nn.Module):
    def __init__(
        self,
        alpha: list[float] | None = None,
        gamma: float = FL_GAMMA,
        label_smoothing: float = 0.05,
        reduction: str = 'mean',
    ):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.reduction       = reduction
        if alpha is not None:
            self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce     = F.cross_entropy(
            logits, targets,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )                             
        pt     = torch.exp(-ce)        
        focal  = (1.0 - pt) ** self.gamma * ce

        if self.alpha is not None:
            self.alpha = self.alpha.to(targets.device)  # ✅ FIX
            focal = self.alpha[targets] * focal

        return focal.mean() if self.reduction == 'mean' else focal.sum()


criterion = FocalLoss(
    alpha=[FL_ALPHA, 1.0 - FL_ALPHA],  
    gamma=FL_GAMMA,
)
print(f'FocalLoss ready (alpha=[{FL_ALPHA}, {1-FL_ALPHA}], gamma={FL_GAMMA})')

In [ ]:
def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    scaler: GradScaler,
    accum_steps: int = ACCUM_STEPS,
) -> tuple[float, float]:
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, (frames, labels) in enumerate(loader):
        frames = frames.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            logits = model(frames)
            loss   = criterion(logits, labels) / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * accum_steps * labels.size(0)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []

    for frames, labels in loader:
        frames = frames.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            logits = model(frames)
            loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        all_probs.append(F.softmax(logits, dim=1).cpu())
        all_labels.append(labels.cpu())

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    return total_loss / len(loader.dataset), probs, labels


def compute_metrics(probs: np.ndarray, labels: np.ndarray, threshold: float = 0.5) -> dict:
    preds = (probs[:, 1] >= threshold).astype(int)
    acc = accuracy_score(labels, preds)
    f1_mac = f1_score(labels, preds, average='macro', zero_division=0)
    f1_dist = f1_score(labels, preds, pos_label=1,     zero_division=0)
    try:
        auc = roc_auc_score(labels, probs[:, 1])
    except Exception:
        auc = float('nan')
    return {'acc': acc, 'macro_f1': f1_mac, 'dist_f1': f1_dist, 'auc': auc}


print('Training functions defined.')

## 8 · Two-Phase Training Loop

In [ ]:
set_seed()
model = EfficientNetBiLSTM().to(DEVICE)
scaler = GradScaler(enabled=USE_AMP)

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'val_macro_f1': [], 'val_dist_f1': [],
    'phase': [],
}


print('\n' + '='*65)
print('PHASE 1 — Frozen EfficientNet backbone')
print('='*65)

model.freeze_backbone()

trainable_p1 = [p for p in model.parameters() if p.requires_grad]
optimizer_p1 = optim.AdamW(trainable_p1, lr=LR_HEAD, weight_decay=1e-4)
scheduler_p1 = optim.lr_scheduler.OneCycleLR(
    optimizer_p1,
    max_lr=LR_HEAD,
    steps_per_epoch=len(train_loader),
    epochs=PHASE1_EPOCHS,
    pct_start=0.3,
)

best_val_loss_p1 = float('inf')
best_state_p1    = None

for epoch in range(1, PHASE1_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer_p1, criterion, scaler)
    scheduler_p1.step()
    val_loss, val_probs, val_labels = eval_epoch(model, val_loader, criterion)
    val_m = compute_metrics(val_probs, val_labels)
    elapsed = time.time() - t0

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_m['acc'])
    history['val_macro_f1'].append(val_m['macro_f1'])
    history['val_dist_f1'].append(val_m['dist_f1'])
    history['phase'].append(1)

    if val_loss < best_val_loss_p1:
        best_val_loss_p1 = val_loss
        best_state_p1    = {k: v.clone() for k, v in model.state_dict().items()}

    print(f'[P1] Ep {epoch:02d}/{PHASE1_EPOCHS}  '
          f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f}  |  '
          f'val_loss={val_loss:.4f} val_acc={val_m["acc"]:.3f}  '
          f'macroF1={val_m["macro_f1"]:.3f} distF1={val_m["dist_f1"]:.3f}  '
          f'[{elapsed:.0f}s]')


model.load_state_dict(best_state_p1)
print(f'\nPhase 1 complete. Best val loss: {best_val_loss_p1:.4f}')

In [ ]:
print('\n' + '='*65)
print('PHASE 2 — Full network fine-tuning')
print('='*65)

model.unfreeze_backbone()


backbone_params = list(model.backbone.parameters())
other_params = [p for p in model.parameters()
                   if id(p) not in {id(q) for q in backbone_params}]

optimizer_p2 = optim.AdamW([
    {'params': backbone_params, 'lr': LR_FULL},
    {'params': other_params,    'lr': LR_FULL * 10},
], weight_decay=1e-4)

scheduler_p2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p2, mode='min', patience=3, factor=0.5, min_lr=1e-7
)

best_val_loss = float('inf')
best_state = None
no_improve = 0

for epoch in range(1, PHASE2_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer_p2, criterion, scaler)
    val_loss, val_probs, val_labels = eval_epoch(model, val_loader, criterion)
    val_m = compute_metrics(val_probs, val_labels)
    elapsed = time.time() - t0

    scheduler_p2.step(val_loss)
    lr_now = optimizer_p2.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_m['acc'])
    history['val_macro_f1'].append(val_m['macro_f1'])
    history['val_dist_f1'].append(val_m['dist_f1'])
    history['phase'].append(2)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
        tag = ' ← best'
    else:
        no_improve += 1
        tag = ''

    print(f'[P2] Ep {epoch:02d}/{PHASE2_EPOCHS}  '
          f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f}  |  '
          f'val_loss={val_loss:.4f} val_acc={val_m["acc"]:.3f}  '
          f'macroF1={val_m["macro_f1"]:.3f} distF1={val_m["dist_f1"]:.3f}  '
          f'lr={lr_now:.1e}  [{elapsed:.0f}s]{tag}')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping triggered at epoch {epoch}.')
        break

model.load_state_dict(best_state)
print(f'\nPhase 2 complete. Best val loss: {best_val_loss:.4f}')

In [ ]:
_, val_probs_final, val_labels_final = eval_epoch(model, val_loader, criterion)

best_thresh, best_dist_f1 = 0.5, 0.0
thresh_grid = np.linspace(0.05, 0.95, 91)
f1_by_thresh = []

for t in thresh_grid:
    preds = (val_probs_final[:, 1] >= t).astype(int)
    f1 = f1_score(val_labels_final, preds, pos_label=1, zero_division=0)
    f1_by_thresh.append(f1)
    if f1 > best_dist_f1:
        best_dist_f1 = f1
        best_thresh = t

print(f'Optimal threshold : {best_thresh:.2f}')
print(f'Distracted F1 (val) : {best_dist_f1:.4f}')

# Also report full metrics at best threshold
val_m_tuned = compute_metrics(val_probs_final, val_labels_final, best_thresh)
print(f'Val Accuracy (tuned): {val_m_tuned["acc"]:.4f}')
print(f'Val Macro F1 (tuned): {val_m_tuned["macro_f1"]:.4f}')
print(f'Val AUC             : {val_m_tuned["auc"]:.4f}')

In [ ]:
_, test_probs, test_labels = eval_epoch(model, test_loader, criterion)
test_preds = (test_probs[:, 1] >= best_thresh).astype(int)

print('\n' + '='*65)
print('TEST SET RESULTS')
print('='*65)
print(classification_report(test_labels, test_preds,target_names=CLASSES, digits=4))

test_m = compute_metrics(test_probs, test_labels, best_thresh)
print(f'Accuracy   : {test_m["acc"]:.4f}')
print(f'Macro F1   : {test_m["macro_f1"]:.4f}')
print(f'Distracted F1: {test_m["dist_f1"]:.4f}')
print(f'ROC-AUC    : {test_m["auc"]:.4f}')

In [ ]:
print("hello")

In [ ]:
epochs_p1 = [i for i, p in enumerate(history['phase'], 1) if p == 1]
epochs_p2 = [i for i, p in enumerate(history['phase'], 1) if p == 2]
all_epochs = list(range(1, len(history['train_loss']) + 1))

fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(2, 3, hspace=0.38, wspace=0.35)


ax = fig.add_subplot(gs[0, 0])
ax.plot(all_epochs, history['train_loss'], color='steelblue', label='Train')
ax.plot(all_epochs, history['val_loss'], color='coral', label='Val')
if epochs_p1 and epochs_p2:
    ax.axvline(max(epochs_p1) + 0.5, color='gray', linestyle='--', alpha=0.6,label='Phase 1→2')
ax.set_title('Loss Curves'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(fontsize=9)


ax = fig.add_subplot(gs[0, 1])
ax.plot(all_epochs, history['train_acc'], color='steelblue', label='Train Acc')
ax.plot(all_epochs, history['val_acc'],   color='coral', label='Val Acc')
if epochs_p1 and epochs_p2:
    ax.axvline(max(epochs_p1) + 0.5, color='gray', linestyle='--', alpha=0.6)
ax.set_title('Accuracy Curves'); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1); ax.legend(fontsize=9)


ax = fig.add_subplot(gs[0, 2])
ax.plot(all_epochs, history['val_macro_f1'], color='darkorange', label='Macro F1')
ax.plot(all_epochs, history['val_dist_f1'], color='purple', label='Distracted F1')
if epochs_p1 and epochs_p2:
    ax.axvline(max(epochs_p1) + 0.5, color='gray', linestyle='--', alpha=0.6)
ax.set_title('Val F1 Scores'); ax.set_xlabel('Epoch'); ax.set_ylabel('F1')
ax.set_ylim(0, 1); ax.legend(fontsize=9)


ax = fig.add_subplot(gs[1, 0])
cm = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(
    cm_norm, annot=cm, fmt='d', cmap='Blues',
    xticklabels=CLASSES, yticklabels=CLASSES,
    ax=ax, cbar=False
)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Test)')


ax = fig.add_subplot(gs[1, 1])
ax.plot(thresh_grid, f1_by_thresh, color='teal', linewidth=2)
ax.axvline(best_thresh, color='red', linestyle='--',
           label=f'Best T={best_thresh:.2f} → F1={best_dist_f1:.3f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Distracted F1 (val)')
ax.set_title('Threshold Sweep'); ax.legend(fontsize=9)


ax = fig.add_subplot(gs[1, 2])
for lbl, cls, col in [(0, 'Engaged', 'steelblue'), (1, 'Distracted', 'coral')]:
    mask = test_labels == lbl
    ax.hist(test_probs[mask, 1], bins=30, alpha=0.6,
            label=cls, color=col, density=True)
ax.axvline(best_thresh, color='red', linestyle='--', label=f'T={best_thresh:.2f}')
ax.set_xlabel('P(Distracted)'); ax.set_ylabel('Density')
ax.set_title('Score Distribution (Test)'); ax.legend(fontsize=9)

plt.suptitle('DAiSEE v2 — EfficientNet-B3 + BiLSTM + Attention', fontsize=14, y=1.01)
plt.savefig('daisee_v2_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved → daisee_v2_results.png')

In [ ]:
metrics_df = pd.DataFrame({
    'epoch':      all_epochs,
    'phase':      history['phase'],
    'train_loss': [f'{v:.4f}' for v in history['train_loss']],
    'val_loss':   [f'{v:.4f}' for v in history['val_loss']],
    'train_acc':  [f'{v:.4f}' for v in history['train_acc']],
    'val_acc':    [f'{v:.4f}' for v in history['val_acc']],
    'val_macro_f1': [f'{v:.4f}' for v in history['val_macro_f1']],
    'val_dist_f1':  [f'{v:.4f}' for v in history['val_dist_f1']],
})
metrics_df.to_csv('training_history.csv', index=False)
print(metrics_df.to_string(index=False))

In [ ]:
checkpoint = {
    'model_state':    model.state_dict(),
    'model_config': {
        'n_classes':   2,
        'lstm_hidden': LSTM_HIDDEN,
        'lstm_layers': LSTM_LAYERS,
        'dropout':     DROPOUT,
    },
    'best_threshold': float(best_thresh),
    'n_frames':       N_FRAMES,
    'img_size':       IMG_SIZE,
    'classes':        CLASSES,
    'test_results':   {k: float(v) for k, v in test_m.items()},
    'history':        history,
}
torch.save(checkpoint, 'daisee_v2.pt')
print('Checkpoint saved → daisee_v2.pt')

# Reload test
ck = torch.load('daisee_v2.pt', map_location='cpu')
print(f'Reloaded OK | threshold={ck["best_threshold"]:.2f} | '
      f'test_acc={ck["test_results"]["acc"]:.4f} | '
      f'test_macroF1={ck["test_results"]["macro_f1"]:.4f}')


def load_model_from_checkpoint(ckpt_path: str, device=None) -> tuple:
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ck = torch.load(ckpt_path, map_location=device)
    m  = EfficientNetBiLSTM(**ck['model_config']).to(device)
    m.load_state_dict(ck['model_state'])
    m.eval()
    return m, ck['best_threshold'], ck['model_config']


print('\nInference helper `load_model_from_checkpoint` ready.')